# Feature Selection for the atmospheric inputs

In [ ]:
import logging
import keras_tuner
import keras
import tensorflow as tf
import time
import pathlib
import os



from usl_models.atmo_ml.model import AtmoModel
from usl_models.atmo_ml import dataset, visualizer, vars

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

logging.getLogger().setLevel(logging.WARNING)
keras.utils.set_random_seed(812)
visualizer.init_plt()


In [ ]:
import keras
from keras import layers
import tensorflow as tf
import numpy as np
import pandas as pd

class MaskedResidualT2SpatialRefiner(keras.Model):
    def __init__(self, st_mask, spatial_mask, use_lu=True):
        super().__init__()

        # save masks as tensors
        self.st_mask = tf.constant(st_mask, dtype=tf.float32)          # [12]
        self.spatial_mask = tf.constant(spatial_mask, dtype=tf.float32)  # [22]
        self.use_lu = use_lu

        # main tower
        self.conv1 = layers.Conv2D(64, 3, padding="same", activation="relu")
        self.conv2 = layers.Conv2D(64, 3, padding="same", activation="relu")
        self.conv3 = layers.Conv2D(32, 3, padding="same", activation="relu")
        self.conv4 = layers.Conv2D(32, 3, padding="same", activation="relu")

        # detail branch
        self.detail_conv1 = layers.Conv2D(32, 3, padding="same", activation="relu")
        self.detail_conv2 = layers.Conv2D(32, 3, padding="same", activation="relu")

        # output heads
        self.residual_out_t0 = layers.Conv2D(1, 1, padding="same", activation="linear")
        self.residual_out_t1 = layers.Conv2D(1, 1, padding="same", activation="linear")

        self.raw_scale_t0 = self.add_weight(
            name="raw_scale_t0", shape=(), initializer="zeros", trainable=True
        )
        self.raw_scale_t1 = self.add_weight(
            name="raw_scale_t1", shape=(), initializer="zeros", trainable=True
        )

        self.avg_pool = layers.AveragePooling2D(pool_size=5, strides=1, padding="same")

    def _normalize_hw(self, x):
        mean = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        std = tf.math.reduce_std(x, axis=[1, 2], keepdims=True) + 1e-6
        return (x - mean) / std

    def _high_pass(self, x):
        smooth = self.avg_pool(x)
        return x - smooth

    def call(self, inputs):
        tt_idx = vars.Spatiotemporal.TT.value

        # -----------------------------
        # Baseline always uses original TT
        # -----------------------------
        tt = inputs["spatiotemporal"][:, :, :, :, tt_idx]  # [B,T,H,W]
        base_t0 = tt[:, -2, :, :][:, tf.newaxis, :, :, tf.newaxis]
        base_t1 = tt[:, -1, :, :][:, tf.newaxis, :, :, tf.newaxis]

        # -----------------------------
        # Masked refinement inputs
        # -----------------------------
        st = inputs["spatiotemporal"]  # [B,T,H,W,12]
        st = st * self.st_mask[tf.newaxis, tf.newaxis, tf.newaxis, tf.newaxis, :]
        st = tf.transpose(st, [0, 2, 3, 1, 4])  # [B,H,W,T,C]
        st = tf.reshape(st, [tf.shape(st)[0], tf.shape(st)[1], tf.shape(st)[2], -1])
        st = self._normalize_hw(st)

        spatial = inputs["spatial"]  # [B,H,W,22]
        spatial = spatial * self.spatial_mask[tf.newaxis, tf.newaxis, tf.newaxis, :]
        spatial = self._normalize_hw(spatial)

        if self.use_lu:
            lu = tf.cast(inputs["lu_index"], tf.float32)[..., tf.newaxis]
            lu = self._normalize_hw(lu)
        else:
            lu = tf.zeros_like(tf.cast(inputs["lu_index"], tf.float32))[..., tf.newaxis]

        fused = tf.concat([st, spatial, lu], axis=-1)

        # main tower
        x = self.conv1(fused)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)

        # high-pass detail branch
        detail_in = tf.concat([
            self._high_pass(st),
            self._high_pass(spatial),
            self._high_pass(lu)
        ], axis=-1)

        d = self.detail_conv1(detail_in)
        d = self.detail_conv2(d)

        x = tf.concat([x, d], axis=-1)

        # residuals
        res_t0 = self.residual_out_t0(x)[:, tf.newaxis, :, :, :]
        res_t1 = self.residual_out_t1(x)[:, tf.newaxis, :, :, :]

        # small bounded scales
        scale_t0 = 0.01 * tf.sigmoid(self.raw_scale_t0)
        scale_t1 = 0.01 * tf.sigmoid(self.raw_scale_t1)

        pred_t0 = base_t0 + scale_t0 * res_t0
        pred_t1 = base_t1 + scale_t1 * res_t1

        return tf.concat([pred_t0, pred_t1], axis=1)

In [ ]:
def t2_loss():
    return keras.losses.MeanAbsoluteError()

def compute_dataset_metrics(model, ds_eval):
    preds = []
    labels = []

    for x_batch, y_batch in ds_eval:
        p = model.predict(x_batch, verbose=0)
        preds.append(p)
        labels.append(y_batch.numpy())

    preds = np.concatenate(preds, axis=0)
    labels = np.concatenate(labels, axis=0)

    mse = np.mean((preds - labels) ** 2)
    mae = np.mean(np.abs(preds - labels))
    return {"mse": float(mse), "mae": float(mae)}

In [ ]:
ST_FEATURES = [v.name for v in vars.Spatiotemporal]   # 12 names
SPATIAL_FEATURES = [f"spatial_{i}" for i in range(22)]
LU_FEATURE = ["lu_index"]

ALL_FEATURES = (
    [("st", i, ST_FEATURES[i]) for i in range(len(ST_FEATURES))] +
    [("spatial", i, SPATIAL_FEATURES[i]) for i in range(len(SPATIAL_FEATURES))] +
    [("lu", 0, "lu_index")]
)

print("Spatiotemporal:", ST_FEATURES)
print("Spatial:", SPATIAL_FEATURES)
print("LU:", LU_FEATURE)

In [ ]:
# This will remove UCPs
ST_FEATURES = [v.name for v in vars.Spatiotemporal]   # 12 names
SPATIAL_FEATURES = [f"spatial_{i}" for i in range(22)]
LU_FEATURE = ["lu_index"]

# remove spatial_3 through spatial_17 inclusive
EXCLUDED_SPATIAL_IDXS = set(range(3, 18))

ALL_FEATURES = (
    [("st", i, ST_FEATURES[i]) for i in range(len(ST_FEATURES))] +
    [
        ("spatial", i, SPATIAL_FEATURES[i])
        for i in range(len(SPATIAL_FEATURES))
        if i not in EXCLUDED_SPATIAL_IDXS
    ] +
    [("lu", 0, "lu_index")]
)

print("Spatiotemporal:", ST_FEATURES)
print("Spatial candidates:", [name for kind, _, name in ALL_FEATURES if kind == "spatial"])
print("LU:", LU_FEATURE)
print("Total candidate features:", len(ALL_FEATURES))

In [ ]:
def build_feature_masks(selected_features):
    st_mask = np.zeros(len(ST_FEATURES), dtype=np.float32)
    spatial_mask = np.zeros(len(SPATIAL_FEATURES), dtype=np.float32)
    use_lu = False

    for kind, idx, name in selected_features:
        if kind == "st":
            st_mask[idx] = 1.0
        elif kind == "spatial":
            spatial_mask[idx] = 1.0
        elif kind == "lu":
            use_lu = True

    return st_mask, spatial_mask, use_lu

In [ ]:
def run_feature_subset(
    selected_features,
    train_ds,
    val_ds_fit,
    val_ds_eval,
    epochs=10,
    steps_per_epoch=3,
    validation_steps=1,
    learning_rate=1e-3,
    verbose=0,
):
    st_mask, spatial_mask, use_lu = build_feature_masks(selected_features)

    model = MaskedResidualT2SpatialRefiner(
        st_mask=st_mask,
        spatial_mask=spatial_mask,
        use_lu=use_lu,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=t2_loss(),
        metrics=[
            keras.metrics.MeanAbsoluteError(),
            keras.metrics.RootMeanSquaredError(),
        ],
        run_eagerly=True,   # stable for notebook/debug
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds_fit,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        verbose=verbose,
    )

    metrics = compute_dataset_metrics(model, val_ds_eval)

    learned_scale_t0 = float((0.01 * tf.sigmoid(model.raw_scale_t0)).numpy())
    learned_scale_t1 = float((0.01 * tf.sigmoid(model.raw_scale_t1)).numpy())

    return {
        "model": model,
        "history": history,
        "selected_features": [name for _, _, name in selected_features],
        "metrics": metrics,
        "scale_t0": learned_scale_t0,
        "scale_t1": learned_scale_t1,
    }

In [ ]:
def greedy_forward_feature_selection(
    train_ds,
    val_ds_fit,
    val_ds_eval,
    candidate_features=ALL_FEATURES,
    max_features=10,
    min_improvement=1e-5,
    epochs=10,
    steps_per_epoch=3,
    validation_steps=1,
):
    selected = []
    remaining = candidate_features.copy()
    search_log = []

    # baseline: no refinement inputs
    baseline_result = run_feature_subset(
        selected_features=selected,
        train_ds=train_ds,
        val_ds_fit=val_ds_fit,
        val_ds_eval=val_ds_eval,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        verbose=0,
    )
    best_score = baseline_result["metrics"]["mse"]
    best_result = baseline_result

    search_log.append({
        "step": 0,
        "added_feature": None,
        "selected_features": [],
        "val_mse": best_score,
        "val_mae": baseline_result["metrics"]["mae"],
        "scale_t0": baseline_result["scale_t0"],
        "scale_t1": baseline_result["scale_t1"],
    })

    print(f"[step 0] baseline mse={best_score:.8f} mae={baseline_result['metrics']['mae']:.8f}")

    for step in range(1, max_features + 1):
        trial_results = []

        for feat in remaining:
            trial_selected = selected + [feat]

            result = run_feature_subset(
                selected_features=trial_selected,
                train_ds=train_ds,
                val_ds_fit=val_ds_fit,
                val_ds_eval=val_ds_eval,
                epochs=epochs,
                steps_per_epoch=steps_per_epoch,
                validation_steps=validation_steps,
                verbose=0,
            )

            trial_results.append((feat, result))

        # choose best feature addition
        feat_best, result_best = min(trial_results, key=lambda x: x[1]["metrics"]["mse"])
        new_score = result_best["metrics"]["mse"]
        improvement = best_score - new_score

        print(
            f"[step {step}] best add={feat_best[2]} "
            f"mse={new_score:.8f} mae={result_best['metrics']['mae']:.8f} "
            f"improvement={improvement:.8f}"
        )

        if improvement < min_improvement:
            print("Stopping: no meaningful improvement.")
            break

        selected.append(feat_best)
        remaining.remove(feat_best)

        best_score = new_score
        best_result = result_best

        search_log.append({
            "step": step,
            "added_feature": feat_best[2],
            "selected_features": [name for _, _, name in selected],
            "val_mse": result_best["metrics"]["mse"],
            "val_mae": result_best["metrics"]["mae"],
            "scale_t0": result_best["scale_t0"],
            "scale_t1": result_best["scale_t1"],
        })

    log_df = pd.DataFrame(search_log)
    return best_result, log_df

In [ ]:
filecache_dir = pathlib.Path("/home/shared/climateiq/filecache")
small_example_keys = [
    ("NYC_Heat_Test/NYC_summer_2000_01p", "2000-05-25"),
    ("NYC_Heat_Test/NYC_summer_2000_01p", "2000-05-26"),
    ("NYC_Heat_Test/NYC_summer_2000_01p", "2000-05-27"),
    ("NYC_Heat_Test/NYC_summer_2000_01p", "2000-05-28"),
    ("NYC_Heat_Test/NYC_summer_2015_50p", "2015-05-25"),
    ("NYC_Heat_Test/NYC_summer_2015_50p", "2015-05-26"),
    ("NYC_Heat_Test/NYC_summer_2018_75p", "2018-05-25"),
    ("NYC_Heat_Test/NYC_summer_2018_75p", "2018-05-26"),
    ("NYC_Heat_Test/NYC_summer_2010_99p", "2010-05-25"),
    ("NYC_Heat_Test/NYC_summer_2010_99p", "2010-05-26"),
    ("NYC_Heat_Test/NYC_summer_2017_25p", "2017-05-25"),
    ("NYC_Heat_Test/NYC_summer_2017_25p", "2017-05-26"),
    # ("PHX_Heat_Test/PHX_summer_2008_25p", "2008-05-25"),
    # ("PHX_Heat_Test/PHX_summer_2008_25p", "2008-05-26"),
    # ("PHX_Heat_Test/PHX_summer_2008_25p", "2008-05-27"),
    # ("PHX_Heat_Test/PHX_summer_2008_25p", "2008-05-28"),
]

train_keys = small_example_keys[:6]
val_keys = small_example_keys[6:]
ds_config_t2 = dataset.Config(
    output_timesteps=2,
    sto_vars=(vars.SpatiotemporalOutput.T2,),
)

# fit datasets (small proxy)
train_ds_t2_search = dataset.load_dataset_cached(
    filecache_dir=filecache_dir,
    example_keys=train_keys,
    config=ds_config_t2,
    shuffle=True,
).batch(2).repeat()

val_ds_t2_fit = dataset.load_dataset_cached(
    filecache_dir=filecache_dir,
    example_keys=val_keys,
    config=ds_config_t2,
    shuffle=False,
).batch(2).repeat()

# real evaluation dataset (no repeat)
val_ds_t2_eval = dataset.load_dataset_cached(
    filecache_dir=filecache_dir,
    example_keys=val_keys,
    config=ds_config_t2,
    shuffle=False,
).batch(1)

In [ ]:
import pathlib
from datetime import date, timedelta

from usl_models.atmo_ml import dataset
from usl_models.atmo_ml import vars

filecache_dir = pathlib.Path("/home/shared/climateiq/filecache")

sim_names = [
    "NYC_Heat_Test/NYC_summer_2000_01p",
    "NYC_Heat_Test/NYC_summer_2010_99p",
    "NYC_Heat_Test/NYC_summer_2015_50p",
    "NYC_Heat_Test/NYC_summer_2017_25p",
    "NYC_Heat_Test/NYC_summer_2018_75p",
    "PHX_Heat_Test/PHX_summer_2008_25p",
    "PHX_Heat_Test/PHX_summer_2009_50p",
    "PHX_Heat_Test/PHX_summer_2011_99p",
    "PHX_Heat_Test/PHX_summer_2015_75p",
    "PHX_Heat_Test/PHX_summer_2020_01p",
]


def get_year(sim_name):
    return int(sim_name.split("_summer_")[1].split("_")[0])


def month_dates(year, month):
    start = date(year, month, 1)

    if month == 12:
        end = date(year + 1, 1, 1)
    else:
        end = date(year, month + 1, 1)

    dates = []
    d = start

    while d < end:
        dates.append(d.strftime("%Y-%m-%d"))
        d += timedelta(days=1)

    return dates


month = 5  # May

full_month_keys = []

for sim_name in sim_names:
    year = get_year(sim_name)

    for day in month_dates(year, month):
        full_month_keys.append((sim_name, day))

print(f"Total examples: {len(full_month_keys)}")
print(full_month_keys[:5])
print(full_month_keys[-5:])

In [ ]:
train_keys = full_month_keys[: int(0.8 * len(full_month_keys))]
val_keys = full_month_keys[int(0.8 * len(full_month_keys)) :]

ds_config_t2 = dataset.Config(
    output_timesteps=2,
    sto_vars=(vars.SpatiotemporalOutput.T2,),
)

train_ds_t2_search = dataset.load_dataset_cached(
    filecache_dir=filecache_dir,
    example_keys=train_keys,
    config=ds_config_t2,
    shuffle=True,
).batch(2).repeat()

val_ds_t2_fit = dataset.load_dataset_cached(
    filecache_dir=filecache_dir,
    example_keys=val_keys,
    config=ds_config_t2,
    shuffle=False,
).batch(2).repeat()

val_ds_t2_eval = dataset.load_dataset_cached(
    filecache_dir=filecache_dir,
    example_keys=val_keys,
    config=ds_config_t2,
    shuffle=False,
).batch(1)

In [ ]:
best_result, search_log_df = greedy_forward_feature_selection(
    train_ds=train_ds_t2_search,
    val_ds_fit=val_ds_t2_fit,
    val_ds_eval=val_ds_t2_eval,
    max_features=8,          # start small
    min_improvement=1e-5,
    epochs=10,               # proxy search budget
    steps_per_epoch=3,
    validation_steps=1,
)

search_log_df

In [ ]:
print("Best selected features:", best_result["selected_features"])
print("Best val MSE:", best_result["metrics"]["mse"])
print("Best val MAE:", best_result["metrics"]["mae"])
print("Learned scales:", best_result["scale_t0"], best_result["scale_t1"])

In [ ]:
winner_feature_names = best_result["selected_features"]
print("Winner feature names:", winner_feature_names)

# rebuild selected tuple list
winner_selected = []
for feat in ALL_FEATURES:
    if feat[2] in winner_feature_names:
        winner_selected.append(feat)

final_result = run_feature_subset(
    selected_features=winner_selected,
    train_ds=train_ds_t2_search,
    val_ds_fit=val_ds_t2_fit,
    val_ds_eval=val_ds_t2_eval,
    epochs=30,
    steps_per_epoch=3,
    validation_steps=1,
    verbose=1,
)

print("Final best subset:", final_result["selected_features"])
print("Final val MSE:", final_result["metrics"]["mse"])
print("Final val MAE:", final_result["metrics"]["mae"])
print("Final scales:", final_result["scale_t0"], final_result["scale_t1"])

In [ ]:
best_model = final_result["model"]

all_preds = []
all_labels = []
all_inputs = []

for x_batch, y_batch in val_ds_t2_eval:
    pred_batch = best_model.predict(x_batch, verbose=0)
    all_preds.append(pred_batch)
    all_labels.append(y_batch.numpy())
    all_inputs.append(x_batch)

all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

In [ ]:
tt_idx = vars.Spatiotemporal.TT.value
all_baselines = []

for x_batch, _ in val_ds_t2_eval:
    tt = x_batch["spatiotemporal"][:, :, :, :, tt_idx]
    base_t0 = tt[:, -2, :, :]
    base_t1 = tt[:, -1, :, :]
    baseline = tf.stack([base_t0, base_t1], axis=1)[..., tf.newaxis]
    all_baselines.append(baseline.numpy())

all_baselines = np.concatenate(all_baselines, axis=0)

In [ ]:
import matplotlib.pyplot as plt

def plot_triplet(labels, baselines, preds, example_idx=0, timestep=0):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].imshow(labels[example_idx, timestep, :, :, 0], origin="lower")
    axes[0].set_title(f"label ex={example_idx} t={timestep}")

    axes[1].imshow(baselines[example_idx, timestep, :, :, 0], origin="lower")
    axes[1].set_title(f"persistence ex={example_idx} t={timestep}")

    axes[2].imshow(preds[example_idx, timestep, :, :, 0], origin="lower")
    axes[2].set_title(f"best subset pred ex={example_idx} t={timestep}")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_triplet(all_labels, all_baselines, all_preds, example_idx=0, timestep=0)
plot_triplet(all_labels, all_baselines, all_preds, example_idx=0, timestep=1)

Now skipping feature selection

In [ ]:
EXCLUDED_SPATIAL_IDXS = set(range(3, 18))  # remove 3–17 inclusive

ST_FEATURES = [v.name for v in vars.Spatiotemporal]
SPATIAL_FEATURES = [f"spatial_{i}" for i in range(22)]
LU_FEATURE = ["lu_index"]

ALL_ALLOWED_FEATURES = (
    [("st", i, ST_FEATURES[i]) for i in range(len(ST_FEATURES))] +
    [
        ("spatial", i, SPATIAL_FEATURES[i])
        for i in range(len(SPATIAL_FEATURES))
        if i not in EXCLUDED_SPATIAL_IDXS
    ] +
    [("lu", 0, "lu_index")]
)

print("Using features:", [name for _, _, name in ALL_ALLOWED_FEATURES])

In [ ]:
final_result = run_feature_subset(
    selected_features=ALL_ALLOWED_FEATURES,
    train_ds=train_ds_t2_search,
    val_ds_fit=val_ds_t2_fit,
    val_ds_eval=val_ds_t2_eval,
    epochs=1500,
    steps_per_epoch=3,
    validation_steps=1,
    verbose=1,
)

print("Final features:", final_result["selected_features"])
print("Final val MSE:", final_result["metrics"]["mse"])
print("Final val MAE:", final_result["metrics"]["mae"])
print("Final scales:", final_result["scale_t0"], final_result["scale_t1"])

In [ ]:
best_model = final_result["model"]

all_preds = []
all_labels = []

for x_batch, y_batch in val_ds_t2_eval:
    pred_batch = best_model.predict(x_batch, verbose=0)
    all_preds.append(pred_batch)
    all_labels.append(y_batch.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

In [ ]:
tt_idx = vars.Spatiotemporal.TT.value

all_baselines = []

for x_batch, _ in val_ds_t2_eval:
    tt = x_batch["spatiotemporal"][:, :, :, :, tt_idx]

    base_t0 = tt[:, -2, :, :]
    base_t1 = tt[:, -1, :, :]

    baseline = tf.stack([base_t0, base_t1], axis=1)[..., tf.newaxis]
    all_baselines.append(baseline.numpy())

all_baselines = np.concatenate(all_baselines, axis=0)

In [ ]:
plot_triplet(all_labels, all_baselines, all_preds, example_idx=0, timestep=0)
plot_triplet(all_labels, all_baselines, all_preds, example_idx=0, timestep=1)

In [ ]:
plot_triplet(all_labels, all_baselines, all_preds, example_idx=0, timestep=0)
plot_triplet(all_labels, all_baselines, all_preds, example_idx=0, timestep=1)

In [ ]:
def plot_error(labels, preds, example_idx=0, timestep=0):
    err = preds - labels

    plt.imshow(err[example_idx, timestep, :, :, 0], origin="lower")
    plt.colorbar()
    plt.title(f"Error (pred - label) ex={example_idx} t={timestep}")
    plt.show()

plot_error(all_labels, all_preds, 0, 0)
plot_error(all_labels, all_preds, 0, 1)

# XGBOOST

In [ ]:
#pip install xgboost

In [ ]:
# ============================================================
# XGBoost feature selection for Atmo ML T2 residual model
# Uses ALL spatial features. Nothing is removed.
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

try:
    from xgboost import XGBRegressor
except ImportError:
    raise ImportError("Please install xgboost first: pip install xgboost")

from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error


# ------------------------------------------------------------
# 1. Define full feature list
# ------------------------------------------------------------

ST_FEATURES = [v.name for v in vars.Spatiotemporal]   # 12
SPATIAL_FEATURES = [f"spatial_{i}" for i in range(22)]
LU_FEATURE = ["lu_index"]

ALL_FEATURES = (
    [("st", i, ST_FEATURES[i]) for i in range(len(ST_FEATURES))] +
    [("spatial", i, SPATIAL_FEATURES[i]) for i in range(len(SPATIAL_FEATURES))] +
    [("lu", 0, "lu_index")]
)

print("Using all candidate features:")
print([name for _, _, name in ALL_FEATURES])

In [ ]:
# ------------------------------------------------------------
# 2. Convert Atmo dataset batches into tabular pixel samples
# ------------------------------------------------------------

def dataset_to_xgb_table(
    ds,
    num_batches,
    max_pixels_per_batch=5000,
    seed=812,
):
    """
    Converts dataset batches into tabular samples.

    Each row = one pixel from one example.
    X includes:
      - all spatiotemporal vars across all input timesteps
      - all 22 spatial vars
      - lu_index

    y includes:
      - T2 target timestep 0
      - T2 target timestep 1
    """

    rng = np.random.default_rng(seed)

    X_list = []
    y_list = []

    feature_names = []

    for batch_idx, (x_batch, y_batch) in enumerate(ds.take(num_batches)):
        st = x_batch["spatiotemporal"].numpy()   # [B,T,H,W,Cst]
        spatial = x_batch["spatial"].numpy()     # [B,H,W,Csp]
        lu = x_batch["lu_index"].numpy()         # [B,H,W]
        y = y_batch.numpy()                      # [B,2,H,W,1]

        B, T, H, W, Cst = st.shape
        Csp = spatial.shape[-1]

        # Move st to [B,H,W,T,Cst], then flatten time/features
        st_flat = np.transpose(st, (0, 2, 3, 1, 4)).reshape(B, H, W, T * Cst)

        # Repeat feature names by timestep
        if batch_idx == 0:
            for t in range(T):
                for c, name in enumerate(ST_FEATURES):
                    feature_names.append(f"{name}_tminus{T - t}")

            for name in SPATIAL_FEATURES:
                feature_names.append(name)

            feature_names.append("lu_index")

        lu_flat = lu[..., np.newaxis]

        X = np.concatenate([st_flat, spatial, lu_flat], axis=-1)  # [B,H,W,F]
        y2 = y[..., 0]                                            # [B,2,H,W]
        y2 = np.transpose(y2, (0, 2, 3, 1))                       # [B,H,W,2]

        X = X.reshape(-1, X.shape[-1])
        y2 = y2.reshape(-1, 2)

        # Drop invalid rows
        valid = np.isfinite(X).all(axis=1) & np.isfinite(y2).all(axis=1)
        X = X[valid]
        y2 = y2[valid]

        # Pixel subsampling to keep XGBoost fast
        if len(X) > max_pixels_per_batch:
            idx = rng.choice(len(X), size=max_pixels_per_batch, replace=False)
            X = X[idx]
            y2 = y2[idx]

        X_list.append(X)
        y_list.append(y2)

    X_all = np.concatenate(X_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)

    return X_all, y_all, feature_names

In [ ]:
# ------------------------------------------------------------
# 3. Build XGBoost training and validation tables
# ------------------------------------------------------------

X_train, y_train, xgb_feature_names = dataset_to_xgb_table(
    train_ds_t2_search,
    num_batches=30,
    max_pixels_per_batch=5000,
)

X_val, y_val, _ = dataset_to_xgb_table(
    val_ds_t2_eval,
    num_batches=10,
    max_pixels_per_batch=10000,
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("Num XGBoost features:", len(xgb_feature_names))

In [ ]:
# ------------------------------------------------------------
# 4. Train XGBoost model
# ------------------------------------------------------------

xgb_base = XGBRegressor(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=812,
    n_jobs=-1,
)

xgb_model = MultiOutputRegressor(xgb_base)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_val)

print("XGBoost val MSE:", mean_squared_error(y_val, y_pred))
print("XGBoost val MAE:", mean_absolute_error(y_val, y_pred))

In [ ]:
# ------------------------------------------------------------
# 5. Extract feature importances
# ------------------------------------------------------------

# Average importance across the two output models: t0 and t1
importances = np.mean(
    [est.feature_importances_ for est in xgb_model.estimators_],
    axis=0,
)

importance_df = pd.DataFrame({
    "xgb_feature": xgb_feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False)

importance_df.head(40)

In [ ]:
# ------------------------------------------------------------
# 6. Aggregate timestep-specific ST importances back to Atmo features
# ------------------------------------------------------------

def aggregate_xgb_importances(importance_df):
    rows = []

    for _, row in importance_df.iterrows():
        fname = row["xgb_feature"]
        imp = row["importance"]

        matched = False

        # Spatiotemporal features appear as NAME_tminusK
        for st_name in ST_FEATURES:
            if fname.startswith(st_name + "_tminus"):
                rows.append({
                    "kind": "st",
                    "idx": ST_FEATURES.index(st_name),
                    "feature": st_name,
                    "importance": imp,
                })
                matched = True
                break

        if matched:
            continue

        # Spatial features
        if fname.startswith("spatial_"):
            idx = int(fname.split("_")[1])
            rows.append({
                "kind": "spatial",
                "idx": idx,
                "feature": fname,
                "importance": imp,
            })
            continue

        # LU
        if fname == "lu_index":
            rows.append({
                "kind": "lu",
                "idx": 0,
                "feature": "lu_index",
                "importance": imp,
            })

    grouped = (
        pd.DataFrame(rows)
        .groupby(["kind", "idx", "feature"], as_index=False)["importance"]
        .sum()
        .sort_values("importance", ascending=False)
    )

    return grouped


grouped_importance_df = aggregate_xgb_importances(importance_df)
grouped_importance_df

In [ ]:
# ------------------------------------------------------------
# 7. Select top K Atmo features from XGBoost ranking
# ------------------------------------------------------------

TOP_K = 5 # change this: 8, 10, 12, 15, etc.

selected_xgb_df = grouped_importance_df.head(TOP_K)

selected_features_xgb = [
    (row.kind, int(row.idx), row.feature)
    for row in selected_xgb_df.itertuples(index=False)
]

print("Selected XGBoost features:")
for feat in selected_features_xgb:
    print(feat)

selected_xgb_df

In [ ]:
# ------------------------------------------------------------
# 8. Train your neural residual model using XGBoost-selected features
# ------------------------------------------------------------

xgb_final_result = run_feature_subset(
    selected_features=selected_features_xgb,
    train_ds=train_ds_t2_search,
    val_ds_fit=val_ds_t2_fit,
    val_ds_eval=val_ds_t2_eval,
    epochs=100,
    steps_per_epoch=3,
    validation_steps=1,
    learning_rate=1e-3,
    verbose=1,
)

print("XGBoost-selected features:", xgb_final_result["selected_features"])
print("Final val MSE:", xgb_final_result["metrics"]["mse"])
print("Final val MAE:", xgb_final_result["metrics"]["mae"])
print("Final scales:", xgb_final_result["scale_t0"], xgb_final_result["scale_t1"])

In [ ]:
# ------------------------------------------------------------
# 9. Plot predictions again
# ------------------------------------------------------------

best_model = xgb_final_result["model"]

all_preds = []
all_labels = []

for x_batch, y_batch in val_ds_t2_eval:
    pred_batch = best_model.predict(x_batch, verbose=0)
    all_preds.append(pred_batch)
    all_labels.append(y_batch.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)


tt_idx = vars.Spatiotemporal.TT.value
all_baselines = []

for x_batch, _ in val_ds_t2_eval:
    tt = x_batch["spatiotemporal"][:, :, :, :, tt_idx]
    base_t0 = tt[:, -2, :, :]
    base_t1 = tt[:, -1, :, :]
    baseline = tf.stack([base_t0, base_t1], axis=1)[..., tf.newaxis]
    all_baselines.append(baseline.numpy())

all_baselines = np.concatenate(all_baselines, axis=0)


plot_triplet(all_labels, all_baselines, all_preds, example_idx=0, timestep=0)
plot_triplet(all_labels, all_baselines, all_preds, example_idx=0, timestep=1)

Now my test

In [ ]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import keras

In [ ]:
# ============================================================
# 1. Define ALL features
# Do NOT remove spatial_3 to spatial_17 here
# ============================================================

ST_FEATURES = [v.name for v in vars.Spatiotemporal]
SPATIAL_FEATURES = [f"spatial_{i}" for i in range(22)]
LU_FEATURE = ["lu_index"]

ALL_FEATURES = (
    [("st", i, ST_FEATURES[i]) for i in range(len(ST_FEATURES))] +
    [("spatial", i, SPATIAL_FEATURES[i]) for i in range(len(SPATIAL_FEATURES))] +
    [("lu", 0, "lu_index")]
)

print("Total candidate features:", len(ALL_FEATURES))
print([name for _, _, name in ALL_FEATURES])

In [ ]:
# ============================================================
# 2. Random feature subset generator
# ============================================================

def make_random_feature_subsets(
    all_features,
    n_trials=6,
    min_features=6,
    max_features=18,
    seed=812,
    always_include_tt=True,
):
    rng = random.Random(seed)

    tt_feature = None
    for feat in all_features:
        kind, idx, name = feat
        if kind == "st" and name == "TT":
            tt_feature = feat
            break

    subsets = []

    for trial in range(n_trials):
        k = rng.randint(min_features, max_features)

        pool = all_features.copy()

        selected = []

        if always_include_tt and tt_feature is not None:
            selected.append(tt_feature)
            pool.remove(tt_feature)
            k = max(k - 1, 0)

        selected += rng.sample(pool, k)

        # keep stable order matching ALL_FEATURES
        selected_set = set(selected)
        selected_ordered = [feat for feat in all_features if feat in selected_set]

        subsets.append(selected_ordered)

    return subsets

In [ ]:
# ============================================================
# 3. Train random feature subsets
# ============================================================

random_subsets = make_random_feature_subsets(
    ALL_FEATURES,
    n_trials=6,
    min_features=6,
    max_features=18,
    seed=812,
    always_include_tt=True,
)

random_results = []

for i, selected_features in enumerate(random_subsets, start=1):
    print("=" * 80)
    print(f"Random trial {i}")
    print("Selected features:")
    print([name for _, _, name in selected_features])

    # reset seed so comparison is less noisy
    keras.utils.set_random_seed(812 + i)

    result = run_feature_subset(
        selected_features=selected_features,
        train_ds=train_ds_t2_search,
        val_ds_fit=val_ds_t2_fit,
        val_ds_eval=val_ds_t2_eval,
        epochs=30,
        steps_per_epoch=3,
        validation_steps=1,
        learning_rate=1e-3,
        verbose=1,
    )

    random_results.append({
        "trial": i,
        "result": result,
        "selected_features": result["selected_features"],
        "mse": result["metrics"]["mse"],
        "mae": result["metrics"]["mae"],
        "scale_t0": result["scale_t0"],
        "scale_t1": result["scale_t1"],
    })

    print(f"Trial {i} MSE:", result["metrics"]["mse"])
    print(f"Trial {i} MAE:", result["metrics"]["mae"])
    print(f"Trial {i} scales:", result["scale_t0"], result["scale_t1"])

In [ ]:
# ============================================================
# 4. Summary dataframe
# ============================================================

random_summary_df = pd.DataFrame([
    {
        "trial": r["trial"],
        "num_features": len(r["selected_features"]),
        "features": r["selected_features"],
        "mse": r["mse"],
        "mae": r["mae"],
        "scale_t0": r["scale_t0"],
        "scale_t1": r["scale_t1"],
    }
    for r in random_results
]).sort_values("mse")

random_summary_df

In [ ]:
# ============================================================
# 5. Collect labels and persistence baseline
# ============================================================

all_labels = []
all_baselines = []

tt_idx = vars.Spatiotemporal.TT.value

for x_batch, y_batch in val_ds_t2_eval:
    all_labels.append(y_batch.numpy())

    tt = x_batch["spatiotemporal"][:, :, :, :, tt_idx]
    base_t0 = tt[:, -2, :, :]
    base_t1 = tt[:, -1, :, :]
    baseline = tf.stack([base_t0, base_t1], axis=1)[..., tf.newaxis]

    all_baselines.append(baseline.numpy())

all_labels = np.concatenate(all_labels, axis=0)
all_baselines = np.concatenate(all_baselines, axis=0)

print("labels:", all_labels.shape)
print("baselines:", all_baselines.shape)

In [ ]:
# ============================================================
# 6. Collect predictions from every random model
# ============================================================

for r in random_results:
    model = r["result"]["model"]

    preds = []

    for x_batch, _ in val_ds_t2_eval:
        pred_batch = model.predict(x_batch, verbose=0)
        preds.append(pred_batch)

    r["preds"] = np.concatenate(preds, axis=0)
    print(f"Trial {r['trial']} preds:", r["preds"].shape)

In [ ]:
# ============================================================
# 7. Plot comparison:
# label | persistence | random trial 1 | random trial 2 | ...
# ============================================================

def plot_random_feature_comparison(
    labels,
    baselines,
    random_results,
    example_idx=0,
    timestep=0,
    sort_by_mse=True,
):
    if sort_by_mse:
        results_to_plot = sorted(random_results, key=lambda r: r["mse"])
    else:
        results_to_plot = random_results

    n_models = len(results_to_plot)
    n_cols = 2 + n_models

    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))

    vmin = np.nanmin(labels[example_idx, timestep, :, :, 0])
    vmax = np.nanmax(labels[example_idx, timestep, :, :, 0])

    axes[0].imshow(
        labels[example_idx, timestep, :, :, 0],
        origin="lower",
        vmin=vmin,
        vmax=vmax,
    )
    axes[0].set_title(f"Label\nex={example_idx}, t={timestep}")
    axes[0].axis("off")

    axes[1].imshow(
        baselines[example_idx, timestep, :, :, 0],
        origin="lower",
        vmin=vmin,
        vmax=vmax,
    )
    axes[1].set_title("Persistence")
    axes[1].axis("off")

    for j, r in enumerate(results_to_plot):
        ax = axes[2 + j]
        preds = r["preds"]

        ax.imshow(
            preds[example_idx, timestep, :, :, 0],
            origin="lower",
            vmin=vmin,
            vmax=vmax,
        )

        ax.set_title(
            f"Trial {r['trial']}\n"
            f"MSE={r['mse']:.4f}\n"
            f"{len(r['selected_features'])} feats"
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Plot both output timesteps
plot_random_feature_comparison(
    all_labels,
    all_baselines,
    random_results,
    example_idx=0,
    timestep=0,
)

plot_random_feature_comparison(
    all_labels,
    all_baselines,
    random_results,
    example_idx=0,
    timestep=1,
)

In [ ]:
# ============================================================
# 8. Error-map comparison
# pred - label
# ============================================================

def plot_random_error_comparison(
    labels,
    random_results,
    example_idx=0,
    timestep=0,
    sort_by_mse=True,
):
    if sort_by_mse:
        results_to_plot = sorted(random_results, key=lambda r: r["mse"])
    else:
        results_to_plot = random_results

    n_models = len(results_to_plot)

    fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4))

    if n_models == 1:
        axes = [axes]

    all_errs = [
        r["preds"][example_idx, timestep, :, :, 0]
        - labels[example_idx, timestep, :, :, 0]
        for r in results_to_plot
    ]

    absmax = max(np.nanmax(np.abs(e)) for e in all_errs)

    for ax, r, err in zip(axes, results_to_plot, all_errs):
        im = ax.imshow(
            err,
            origin="lower",
            vmin=-absmax,
            vmax=absmax,
            cmap="coolwarm",
        )

        ax.set_title(
            f"Trial {r['trial']} error\n"
            f"MSE={r['mse']:.4f}"
        )
        ax.axis("off")

    fig.colorbar(im, ax=axes, shrink=0.8)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_random_error_comparison(
    all_labels,
    random_results,
    example_idx=0,
    timestep=0,
)

plot_random_error_comparison(
    all_labels,
    random_results,
    example_idx=0,
    timestep=1,
)

In [ ]:
# ============================================================
# 9. Print best/worst random feature sets
# ============================================================

best_random = min(random_results, key=lambda r: r["mse"])
worst_random = max(random_results, key=lambda r: r["mse"])

print("BEST RANDOM TRIAL")
print("Trial:", best_random["trial"])
print("MSE:", best_random["mse"])
print("MAE:", best_random["mae"])
print("Features:", best_random["selected_features"])

print("\nWORST RANDOM TRIAL")
print("Trial:", worst_random["trial"])
print("MSE:", worst_random["mse"])
print("MAE:", worst_random["mae"])
print("Features:", worst_random["selected_features"])

In [ ]:
# ============================================================
# End-to-end feature comparison experiment
#
# Models compared:
#   1. ALL_FEATURES
#   2. REMOVE_SPATIAL_3_TO_17
#   3. RANDOM_1 ... RANDOM_6
#
# Model:
#   Simple CNN residual model:
#   prediction = TT persistence baseline + learned CNN residual
# ============================================================

import random
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
import matplotlib.pyplot as plt

from usl_models.atmo_ml import vars as atmo_vars


# ============================================================
# 1. Feature definitions
# ============================================================

ST_FEATURES = [v.name for v in atmo_vars.Spatiotemporal]
SPATIAL_FEATURES = [f"spatial_{i}" for i in range(22)]
LU_FEATURE = ["lu_index"]

ALL_FEATURES = (
    [("st", i, ST_FEATURES[i]) for i in range(len(ST_FEATURES))] +
    [("spatial", i, SPATIAL_FEATURES[i]) for i in range(len(SPATIAL_FEATURES))] +
    [("lu", 0, "lu_index")]
)

print("Total candidate features:", len(ALL_FEATURES))
print([name for _, _, name in ALL_FEATURES])


# ============================================================
# 2. Mask builder
# ============================================================

def build_feature_masks(selected_features):
    st_mask = np.zeros(len(ST_FEATURES), dtype=np.float32)
    spatial_mask = np.zeros(len(SPATIAL_FEATURES), dtype=np.float32)
    use_lu = False

    for kind, idx, name in selected_features:
        if kind == "st":
            st_mask[idx] = 1.0
        elif kind == "spatial":
            spatial_mask[idx] = 1.0
        elif kind == "lu":
            use_lu = True

    return st_mask, spatial_mask, use_lu


# ============================================================
# 3. Simple CNN residual model
# ============================================================

class SimpleMaskedCNNRefiner(keras.Model):
    def __init__(self, st_mask, spatial_mask, use_lu=True):
        super().__init__()

        self.st_mask = tf.constant(st_mask, dtype=tf.float32)
        self.spatial_mask = tf.constant(spatial_mask, dtype=tf.float32)
        self.use_lu = use_lu

        self.conv1 = layers.Conv2D(64, 3, padding="same", activation="relu")
        self.conv2 = layers.Conv2D(64, 3, padding="same", activation="relu")
        self.conv3 = layers.Conv2D(32, 3, padding="same", activation="relu")
        self.conv4 = layers.Conv2D(16, 3, padding="same", activation="relu")

        self.out = layers.Conv2D(2, 1, padding="same", activation="linear")

    def _normalize_hw(self, x):
        mean = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        std = tf.math.reduce_std(x, axis=[1, 2], keepdims=True) + 1e-6
        return (x - mean) / std

    def call(self, inputs):
        tt_idx = atmo_vars.Spatiotemporal.TT.value

        # Persistence baseline
        tt = inputs["spatiotemporal"][:, :, :, :, tt_idx]

        base_t0 = tt[:, -2, :, :]
        base_t1 = tt[:, -1, :, :]

        base_t0 = base_t0[:, tf.newaxis, :, :, tf.newaxis]
        base_t1 = base_t1[:, tf.newaxis, :, :, tf.newaxis]

        # Spatiotemporal inputs
        st = inputs["spatiotemporal"]
        st = st * self.st_mask[tf.newaxis, tf.newaxis, tf.newaxis, tf.newaxis, :]

        st = tf.transpose(st, [0, 2, 3, 1, 4])

        b = tf.shape(st)[0]
        h = tf.shape(st)[1]
        w = tf.shape(st)[2]

        st = tf.reshape(st, [b, h, w, -1])
        st = self._normalize_hw(st)

        # Spatial inputs
        spatial = inputs["spatial"]
        spatial = spatial * self.spatial_mask[tf.newaxis, tf.newaxis, tf.newaxis, :]
        spatial = self._normalize_hw(spatial)

        # LU input
        if self.use_lu:
            lu = tf.cast(inputs["lu_index"], tf.float32)[..., tf.newaxis]
            lu = self._normalize_hw(lu)
        else:
            lu = tf.zeros_like(tf.cast(inputs["lu_index"], tf.float32))[..., tf.newaxis]

        # CNN residual correction
        x = tf.concat([st, spatial, lu], axis=-1)

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)

        residual = self.out(x)

        res_t0 = residual[:, :, :, 0][:, tf.newaxis, :, :, tf.newaxis]
        res_t1 = residual[:, :, :, 1][:, tf.newaxis, :, :, tf.newaxis]

        pred_t0 = base_t0 + res_t0
        pred_t1 = base_t1 + res_t1

        return tf.concat([pred_t0, pred_t1], axis=1)


# ============================================================
# 4. Metric helper
# ============================================================

def compute_dataset_metrics(model, ds_eval):
    preds = []
    labels = []

    for x_batch, y_batch in ds_eval:
        p = model.predict(x_batch, verbose=0)
        preds.append(p)
        labels.append(y_batch.numpy())

    preds = np.concatenate(preds, axis=0)
    labels = np.concatenate(labels, axis=0)

    return {
        "mse": float(np.mean((preds - labels) ** 2)),
        "mae": float(np.mean(np.abs(preds - labels))),
    }


# ============================================================
# 5. Train one subset
# ============================================================

def run_simple_cnn_subset(
    selected_features,
    train_ds,
    val_ds_fit,
    val_ds_eval,
    epochs=20,
    steps_per_epoch=3,
    validation_steps=1,
    learning_rate=3e-4,
    verbose=1,
):
    st_mask, spatial_mask, use_lu = build_feature_masks(selected_features)

    model = SimpleMaskedCNNRefiner(
        st_mask=st_mask,
        spatial_mask=spatial_mask,
        use_lu=use_lu,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=keras.losses.MeanAbsoluteError(),
        metrics=[
            keras.metrics.MeanAbsoluteError(),
            keras.metrics.RootMeanSquaredError(),
        ],
        run_eagerly=True,
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds_fit,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        verbose=verbose,
    )

    metrics = compute_dataset_metrics(model, val_ds_eval)

    return {
        "model": model,
        "history": history,
        "selected_features": [name for _, _, name in selected_features],
        "metrics": metrics,
    }


# ============================================================
# 6. Random feature subset generator
# ============================================================

def make_random_feature_subsets(
    all_features,
    n_trials=6,
    min_features=8,
    max_features=20,
    seed=812,
    always_include_tt=True,
):
    rng = random.Random(seed)

    tt_feature = None
    for feat in all_features:
        if feat[0] == "st" and feat[2] == "TT":
            tt_feature = feat
            break

    subsets = []

    for _ in range(n_trials):
        k = rng.randint(min_features, max_features)

        pool = all_features.copy()
        selected = []

        if always_include_tt and tt_feature is not None:
            selected.append(tt_feature)
            pool.remove(tt_feature)
            k = k - 1

        selected += rng.sample(pool, k)

        selected_set = set(selected)
        selected_ordered = [feat for feat in all_features if feat in selected_set]

        subsets.append(selected_ordered)

    return subsets


# ============================================================
# 7. Build experiment subsets
# ============================================================

random_subsets = make_random_feature_subsets(
    ALL_FEATURES,
    n_trials=6,
    min_features=8,
    max_features=20,
    seed=812,
    always_include_tt=True,
)

ALL_FEATURES_SUBSET = ALL_FEATURES.copy()

EXCLUDED_SPATIAL_IDXS = set(range(3, 18))

REMOVE_SPATIAL_3_TO_17_SUBSET = [
    feat for feat in ALL_FEATURES
    if not (feat[0] == "spatial" and feat[1] in EXCLUDED_SPATIAL_IDXS)
]

experiment_subsets = [
    {
        "name": "ALL_FEATURES",
        "features": ALL_FEATURES_SUBSET,
    },
    {
        "name": "REMOVE_SPATIAL_3_TO_17",
        "features": REMOVE_SPATIAL_3_TO_17_SUBSET,
    },
]

for i, subset in enumerate(random_subsets, start=1):
    experiment_subsets.append({
        "name": f"RANDOM_{i}",
        "features": subset,
    })

print("\nExperiment subsets:")
for exp in experiment_subsets:
    print(
        exp["name"],
        len(exp["features"]),
        [name for _, _, name in exp["features"]],
    )


# ============================================================
# 8. Train all experiments
# ============================================================

experiment_results = []

for i, exp in enumerate(experiment_subsets, start=1):
    selected_features = exp["features"]
    exp_name = exp["name"]

    print("\n" + "=" * 100)
    print(f"Experiment {i}: {exp_name}")
    print("Selected features:")
    print([name for _, _, name in selected_features])

    keras.utils.set_random_seed(812 + i)

    result = run_simple_cnn_subset(
        selected_features=selected_features,
        train_ds=train_ds_t2_search,
        val_ds_fit=val_ds_t2_fit,
        val_ds_eval=val_ds_t2_eval,
        epochs=20,
        steps_per_epoch=3,
        validation_steps=1,
        learning_rate=3e-4,
        verbose=1,
    )

    experiment_results.append({
        "trial": i,
        "name": exp_name,
        "result": result,
        "selected_features": result["selected_features"],
        "mse": result["metrics"]["mse"],
        "mae": result["metrics"]["mae"],
    })

    print("MSE:", result["metrics"]["mse"])
    print("MAE:", result["metrics"]["mae"])


# ============================================================
# 9. Summary dataframe
# ============================================================

summary_df = pd.DataFrame([
    {
        "trial": r["trial"],
        "name": r["name"],
        "num_features": len(r["selected_features"]),
        "features": r["selected_features"],
        "mse": r["mse"],
        "mae": r["mae"],
    }
    for r in experiment_results
]).sort_values("mse")

display(summary_df)


# ============================================================
# 10. Collect labels and persistence baseline
# ============================================================

all_labels = []
all_baselines = []

tt_idx = atmo_vars.Spatiotemporal.TT.value

for x_batch, y_batch in val_ds_t2_eval:
    all_labels.append(y_batch.numpy())

    tt = x_batch["spatiotemporal"][:, :, :, :, tt_idx]
    base_t0 = tt[:, -2, :, :]
    base_t1 = tt[:, -1, :, :]
    baseline = tf.stack([base_t0, base_t1], axis=1)[..., tf.newaxis]

    all_baselines.append(baseline.numpy())

all_labels = np.concatenate(all_labels, axis=0)
all_baselines = np.concatenate(all_baselines, axis=0)

print("Labels shape:", all_labels.shape)
print("Baseline shape:", all_baselines.shape)


# ============================================================
# 11. Collect predictions
# ============================================================

for r in experiment_results:
    preds = []

    for x_batch, _ in val_ds_t2_eval:
        pred_batch = r["result"]["model"].predict(x_batch, verbose=0)
        preds.append(pred_batch)

    r["preds"] = np.concatenate(preds, axis=0)
    print(f"{r['name']} preds shape:", r["preds"].shape)


# ============================================================
# 12. Plot label | persistence | all experiments
# ============================================================

def plot_experiment_comparison(
    labels,
    baselines,
    experiment_results,
    example_idx=0,
    timestep=0,
    sort_by_mse=True,
):
    if sort_by_mse:
        results_to_plot = sorted(experiment_results, key=lambda r: r["mse"])
    else:
        results_to_plot = experiment_results

    n_cols = 2 + len(results_to_plot)

    fig, axes = plt.subplots(
        1,
        n_cols,
        figsize=(4 * n_cols, 4),
        squeeze=False,
    )
    axes = axes[0]

    vmin = np.nanmin(labels[example_idx, timestep, :, :, 0])
    vmax = np.nanmax(labels[example_idx, timestep, :, :, 0])

    axes[0].imshow(
        labels[example_idx, timestep, :, :, 0],
        origin="lower",
        vmin=vmin,
        vmax=vmax,
    )
    axes[0].set_title(f"Label\nex={example_idx}, t={timestep}")
    axes[0].axis("off")

    axes[1].imshow(
        baselines[example_idx, timestep, :, :, 0],
        origin="lower",
        vmin=vmin,
        vmax=vmax,
    )
    axes[1].set_title("Persistence")
    axes[1].axis("off")

    for j, r in enumerate(results_to_plot):
        ax = axes[2 + j]

        ax.imshow(
            r["preds"][example_idx, timestep, :, :, 0],
            origin="lower",
            vmin=vmin,
            vmax=vmax,
        )

        ax.set_title(
            f"{r['name']}\n"
            f"MSE={r['mse']:.4f}\n"
            f"{len(r['selected_features'])} feats"
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_experiment_comparison(
    all_labels,
    all_baselines,
    experiment_results,
    example_idx=0,
    timestep=0,
)

plot_experiment_comparison(
    all_labels,
    all_baselines,
    experiment_results,
    example_idx=0,
    timestep=1,
)


# ============================================================
# 13. Plot error maps
# ============================================================

def plot_experiment_error_comparison(
    labels,
    experiment_results,
    example_idx=0,
    timestep=0,
    sort_by_mse=True,
):
    if sort_by_mse:
        results_to_plot = sorted(experiment_results, key=lambda r: r["mse"])
    else:
        results_to_plot = experiment_results

    n_models = len(results_to_plot)

    fig, axes = plt.subplots(
        1,
        n_models,
        figsize=(4 * n_models, 4),
        squeeze=False,
    )
    axes = axes[0]

    all_errs = [
        r["preds"][example_idx, timestep, :, :, 0]
        - labels[example_idx, timestep, :, :, 0]
        for r in results_to_plot
    ]

    absmax = max(np.nanmax(np.abs(e)) for e in all_errs)

    for ax, r, err in zip(axes, results_to_plot, all_errs):
        im = ax.imshow(
            err,
            origin="lower",
            vmin=-absmax,
            vmax=absmax,
            cmap="coolwarm",
        )

        ax.set_title(
            f"{r['name']} error\n"
            f"MSE={r['mse']:.4f}"
        )
        ax.axis("off")

    fig.colorbar(im, ax=axes, shrink=0.8)
    plt.tight_layout()
    plt.show()


plot_experiment_error_comparison(
    all_labels,
    experiment_results,
    example_idx=0,
    timestep=0,
)

plot_experiment_error_comparison(
    all_labels,
    experiment_results,
    example_idx=0,
    timestep=1,
)


# ============================================================
# 14. Best and worst experiments
# ============================================================

best_exp = min(experiment_results, key=lambda r: r["mse"])
worst_exp = max(experiment_results, key=lambda r: r["mse"])

print("\nBEST EXPERIMENT")
print("Name:", best_exp["name"])
print("MSE:", best_exp["mse"])
print("MAE:", best_exp["mae"])
print("Features:", best_exp["selected_features"])

print("\nWORST EXPERIMENT")
print("Name:", worst_exp["name"])
print("MSE:", worst_exp["mse"])
print("MAE:", worst_exp["mae"])
print("Features:", worst_exp["selected_features"])